# Manifold Matching via Autoencoders — Colab Pipeline

Reproduces results from **"Manifold Matching via Autoencoders"** (ICML submission).

**Runtime:** ~2–3 h on T4 GPU (skip-HPO mode) · ~12 h with full HPO  
**Recommended:** Runtime → Change runtime type → T4 GPU

---
### Steps
1. Clone the repository from GitHub
2. Mount Google Drive (results only)
3. Install dependencies
4. GPU check & global config
5. Experiment configuration
6. (Optional) Hyperparameter optimisation
7. Final evaluation
8. View results


## 1 — Clone Repository

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/laurent-cheret/manifold-matching-autoencoders.git'
REPO_DIR = '/content/manifold-matching-autoencoders'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'Working directory: {os.getcwd()}')
print('Files:', os.listdir('.')[:8])


## 2 — Mount Google Drive (Results Only)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_RESULTS = '/content/drive/MyDrive/topo_compare_results'
os.makedirs(DRIVE_RESULTS, exist_ok=True)
print(f'Results will be saved to: {DRIVE_RESULTS}')


## 3 — Install Dependencies

In [ ]:
%%bash
pip install -q \
    gudhi \
    optuna \
    scanpy \
    anndata \
    umap-learn \
    git+https://github.com/ArGintum/RipserZeros.git

python -c "import torch; print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())"
python -c "import gudhi; print('gudhi:', gudhi.__version__)"
python -c "import optuna; print('optuna:', optuna.__version__)"


## 4 — GPU Check & Global Config

In [ ]:
import torch, os, sys
sys.path.insert(0, os.getcwd())

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected — training will be slow. Change runtime to GPU.')

SEED = 42

DATA_RAW = os.path.join(os.getcwd(), 'data', 'raw')
os.makedirs(DATA_RAW, exist_ok=True)
print(f'Data cache: {DATA_RAW}')


## 5 — Experiment Configuration

Edit the variables below to choose which experiments to run.

In [ ]:
# ── Datasets ─────────────────────────────────────────────
SYNTHETIC_DATASETS = ['spheres', 'swiss_roll', 'concentric_spheres',
                      'linked_tori', 'branching_tree']
REAL_DATASETS      = ['mnist', 'fmnist', 'cifar10']
BIO_DATASETS       = ['pbmc3k', 'paul15']

# Set to True to include each group
RUN_SYNTHETIC = True
RUN_REAL      = True
RUN_BIO       = False   # requires scanpy; set True if installed

# ── Models ───────────────────────────────────────────────
MODELS = ['vanilla', 'topoae', 'rtdae', 'mmae']  # add 'geomae', 'ggae' if desired

# ── Latent dimensions ──────────────────────────────────────
LATENT_DIMS_SYNTH = [2]          # synthetic: visualisation only needs 2D
LATENT_DIMS_REAL  = [2, 16, 32]  # real-world: sweep dimensions

# ── Training ──────────────────────────────────────────────
SKIP_HPO     = True    # True = use precomputed configs; False = run Optuna HPO
HPO_TRIALS   = 30      # trials per model×dataset (only used if SKIP_HPO=False)
HPO_EPOCHS   = 30
EVAL_EPOCHS  = 100
N_SEEDS      = 3       # seeds for final evaluation (3 is enough for Colab)

print('Configuration saved.')
print(f'Results dir: {DRIVE_RESULTS}')


## 6 — (Optional) Hyperparameter Optimisation

Skip this section if `SKIP_HPO = True` above. Best configs are saved to `configs/best_hparams/` inside the cloned repo.


In [ ]:
import subprocess, sys

if SKIP_HPO:
    print("SKIP_HPO=True — skipping HPO, using precomputed configs.")
else:
    datasets = []
    if RUN_SYNTHETIC: datasets += SYNTHETIC_DATASETS
    if RUN_REAL:      datasets += REAL_DATASETS
    if RUN_BIO:       datasets += BIO_DATASETS

    for dataset in datasets:
        dims = LATENT_DIMS_SYNTH if dataset in SYNTHETIC_DATASETS else LATENT_DIMS_REAL
        for model in MODELS:
            for dim in dims:
                print(f"
>>> HPO: {model} on {dataset} dim={dim}")
                cmd = [
                    sys.executable, "hyperparam_search_optuna.py",
                    "--model",       model,
                    "--dataset",     dataset,
                    "--latent_dim",  str(dim),
                    "--opt_metric",  "density_kl_0_1",
                    "--n_trials",    str(HPO_TRIALS),
                    "--epochs",      str(HPO_EPOCHS),
                    "--device",      DEVICE,
                    "--seed",        str(SEED),
                    "--results_dir", f"{DRIVE_RESULTS}/hypersearch/{model}_{dataset}_dim{dim}",
                ]
                result = subprocess.run(cmd, capture_output=True, text=True)
                print(result.stdout[-2000:] if result.stdout else "")
                if result.returncode != 0:
                    print("STDERR:", result.stderr[-1000:])


## 7 — Final Evaluation

In [ ]:
import subprocess, sys

def run_eval(dataset, latent_dim, output_subdir=None):
    out_dir = output_subdir or f'{DRIVE_RESULTS}/final/{dataset}_dim{latent_dim}'
    best_dir = f'configs/best_hparams/{dataset}'
    cmd = [
        sys.executable, 'run_final_evaluation.py',
        '--dataset',          dataset,
        '--best_configs_dir', best_dir,
        '--output_dir',       out_dir,
        '--latent_dim',       str(latent_dim),
        '--n_seeds',          str(N_SEEDS),
        '--epochs',           str(EVAL_EPOCHS),
        '--device',           DEVICE,
        '--seed',             str(SEED),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout[-3000:] if result.stdout else "")
    if result.returncode != 0:
        print('STDERR:', result.stderr[-1000:])
    return out_dir

# ── Synthetic datasets ───────────────────────────────────────────
if RUN_SYNTHETIC:
    print('='*60, '\nSYNTHETIC DATASETS\n', '='*60)
    for dataset in SYNTHETIC_DATASETS:
        print(f'\n>>> Evaluating {dataset}')
        run_eval(dataset, latent_dim=2)


In [ ]:
# ── Real-world datasets (multiple latent dims) ────────────────────────
if RUN_REAL:
    print('='*60, '\nREAL-WORLD DATASETS\n', '='*60)
    for dataset in REAL_DATASETS:
        for dim in LATENT_DIMS_REAL:
            print(f'\n>>> Evaluating {dataset} (dim={dim})')
            run_eval(dataset, latent_dim=dim)


In [ ]:
# ── Biological datasets ──────────────────────────────────────────
if RUN_BIO:
    print('='*60, '\nBIOLOGICAL DATASETS\n', '='*60)
    for dataset in BIO_DATASETS:
        print(f'\n>>> Evaluating {dataset}')
        run_eval(dataset, latent_dim=2)


## 8 — Quick Single-Run (Debug / Visualise)

Train one model on one dataset and display the 2D latent space.

In [ ]:
DATASET  = 'spheres'
MODEL    = 'mmae'
EPOCHS   = 50
LAT_DIM  = 2

import subprocess, sys
result = subprocess.run([
    sys.executable, 'run_experiment.py',
    '--dataset',     DATASET,
    '--model',       MODEL,
    '--latent_dim',  str(LAT_DIM),
    '--epochs',      str(EPOCHS),
    '--device',      DEVICE,
    '--seed',        str(SEED),
    '--output_dir',  f'{DRIVE_RESULTS}/single_runs',
], capture_output=True, text=True)
print(result.stdout[-3000:] if result.stdout else "")
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])


In [ ]:
import glob
from IPython.display import Image, display

plots = sorted(glob.glob(f'{DRIVE_RESULTS}/single_runs/{DATASET}/{MODEL}/**/latent_space.png', recursive=True))
if plots:
    display(Image(plots[-1]))
else:
    print('No plot found — check if run completed successfully.')


## 9 — View Results

In [ ]:
import pandas as pd, glob, os

rows = []
for f in glob.glob(f'{DRIVE_RESULTS}/final/**/summary.csv', recursive=True):
    rows.append(pd.read_csv(f))

if rows:
    summary = pd.concat(rows, ignore_index=True)
    cols = [c for c in summary.columns if c in [
        'dataset', 'model', 'latent_dim',
        'distance_correlation_mean', 'trustworthiness_10_mean',
        'density_kl_0_1_mean', 'wasserstein_H1_mean'
    ]]
    display(summary[cols].sort_values(['dataset','model']))
else:
    print('No summary.csv files found yet. Run evaluation cells first.')


In [ ]:
import subprocess, sys, glob, os

subprocess.run([
    sys.executable, 'generate_latex_table.py',
    '--results_dir', f'{DRIVE_RESULTS}/final',
    '--output_dir',  f'{DRIVE_RESULTS}/latex',
], check=False)

for f in glob.glob(f'{DRIVE_RESULTS}/latex/*.tex'):
    print(f'\n--- {os.path.basename(f)} ---')
    with open(f) as fh:
        print(fh.read())


## 10 — Rebuttal: W0 vs Latent Dimension

Trains all models with **fixed hyperparameters** (no HPO) across 4 latent dims and 3 seeds, reporting W0 (Wasserstein H0). Shows that distance-preserving methods (MMAE, SPAE) become more competitive as the bottleneck grows.


In [ ]:
import subprocess, sys, json, glob, os
import numpy as np
import pandas as pd

# ── Config ────────────────────────────────────────────────
REBUTTAL_DATASETS = ["mnist", "fmnist", "cifar10"]
REBUTTAL_MODELS   = ["vanilla", "topoae", "rtdae", "spae", "mmae"]
REBUTTAL_DIMS     = [2, 16, 64, 128]
REBUTTAL_SEEDS    = [42, 43, 44]
REBUTTAL_EPOCHS   = 50
REBUTTAL_DIR      = f"{DRIVE_RESULTS}/rebuttal_w0"

FIXED_HPARAMS = {
    "vanilla": {"lr": 1e-4, "batch_size": 256},
    "mmae":    {"lr": 1e-4, "batch_size": 256},
    "spae":    {"lr": 1e-4, "batch_size": 256},
    "topoae":  {"lr": 1e-4, "batch_size": 128},
    "rtdae":   {"lr": 1e-4, "batch_size": 100},
}

# ── Run (with checkpoint resume) ─────────────────────────────────
raw_results = {}  # (dataset, model, dim) -> [w0_seed42, w0_seed43, w0_seed44]

total = len(REBUTTAL_DATASETS) * len(REBUTTAL_MODELS) * len(REBUTTAL_DIMS) * len(REBUTTAL_SEEDS)
done  = 0

for dataset in REBUTTAL_DATASETS:
    for model in REBUTTAL_MODELS:
        hp = FIXED_HPARAMS[model]
        for dim in REBUTTAL_DIMS:
            w0_scores = []
            for seed in REBUTTAL_SEEDS:
                done += 1
                out_dir = f"{REBUTTAL_DIR}/dim{dim}/seed{seed}"
                pattern = f"{out_dir}/{dataset}/{model}/*/metrics.json"
                existing = sorted(glob.glob(pattern))

                if existing:
                    print(f"[{done}/{total}] SKIP (cached) {model} | {dataset} | dim={dim} | seed={seed}", flush=True)
                    with open(existing[-1]) as f:
                        m = json.load(f)
                    w0 = m.get("wasserstein_H0", float("nan"))
                    w0_scores.append(w0)
                    print(f"  W0 = {w0:.4f}")
                    continue

                print(f"[{done}/{total}] {model} | {dataset} | dim={dim} | seed={seed}", flush=True)
                cmd = [
                    sys.executable, "run_experiment.py",
                    "--dataset",    dataset,
                    "--model",      model,
                    "--latent_dim", str(dim),
                    "--epochs",     str(REBUTTAL_EPOCHS),
                    "--batch_size", str(hp["batch_size"]),
                    "--lr",         str(hp["lr"]),
                    "--device",     DEVICE,
                    "--seed",       str(seed),
                    "--output_dir", out_dir,
                ]
                result = subprocess.run(cmd, capture_output=True, text=True)
                if result.returncode != 0:
                    print(f"  ERROR: {result.stderr[-400:]}")
                    w0_scores.append(float("nan"))
                    continue
                files = sorted(glob.glob(pattern))
                if not files:
                    print("  WARNING: metrics.json not found")
                    w0_scores.append(float("nan"))
                    continue
                with open(files[-1]) as f:
                    m = json.load(f)
                w0 = m.get("wasserstein_H0", float("nan"))
                w0_scores.append(w0)
                print(f"  W0 = {w0:.4f}")
            raw_results[(dataset, model, dim)] = w0_scores

print("
All runs complete.")


In [ ]:
# ── Aggregate & display ───────────────────────────────────────────────────
import numpy as np, pandas as pd

rows = []
for dataset in REBUTTAL_DATASETS:
    row = {'dataset': dataset}
    for model in REBUTTAL_MODELS:
        for dim in REBUTTAL_DIMS:
            scores = raw_results.get((dataset, model, dim), [])
            scores = [s for s in scores if not (isinstance(s, float) and s != s)]  # drop nan
            if scores:
                mean = np.mean(scores)
                row[f'{model}_{dim}d'] = f'{mean:.3f}'
            else:
                row[f'{model}_{dim}d'] = 'N/A'
    rows.append(row)

df = pd.DataFrame(rows).set_index('dataset')

# Reorder columns: group by dim
cols = [f'{m}_{d}d' for d in REBUTTAL_DIMS for m in REBUTTAL_MODELS]
df = df[[c for c in cols if c in df.columns]]
display(df)

# ── LaTeX ─────────────────────────────────────────────────────────────────
print('\n--- LaTeX ---')
latex_lines = []
latex_lines.append(r'\begin{tabular}{l' + 'c' * len(df.columns) + r'}')
latex_lines.append(r'\toprule')
header = ' & '.join(['Dataset'] + list(df.columns)) + r' \\\\')
latex_lines.append(header)
latex_lines.append(r'\midrule')
for idx, row in df.iterrows():
    latex_lines.append(' & '.join([idx] + list(row.values)) + r' \\')
latex_lines.append(r'\bottomrule')
latex_lines.append(r'\end{tabular}')
print('\n'.join(latex_lines))

# Save to Drive
os.makedirs(f'{DRIVE_RESULTS}/rebuttal_w0', exist_ok=True)
df.to_csv(f'{DRIVE_RESULTS}/rebuttal_w0/w0_vs_dim_table.csv')
with open(f'{DRIVE_RESULTS}/rebuttal_w0/w0_vs_dim_table.tex', 'w') as f:
    f.write('\n'.join(latex_lines))
print(f'\nSaved to {DRIVE_RESULTS}/rebuttal_w0/')


In [ ]:
# ── MMAE Win-Rate Analysis across all metrics ────────────────────────────────
# Re-reads every metrics.json from Drive so this cell is self-contained.
# For each metric, counts how often MMAE is the best model across all
# (dataset, dim) combos, then shows a ranked win-rate table and lets you
# build a full display table for the metrics you care about.

import glob, json, os
import numpy as np
import pandas as pd

REBUTTAL_DATASETS = ["mnist", "fmnist", "cifar10"]
REBUTTAL_MODELS   = ["vanilla", "topoae", "rtdae", "spae", "mmae"]
REBUTTAL_DIMS     = [2, 16, 64, 128]
REBUTTAL_SEEDS    = [42, 43, 44]
REBUTTAL_DIR      = f"{DRIVE_RESULTS}/rebuttal_w0"

LOWER_IS_BETTER = {
    'wasserstein_H0', 'wasserstein_H1',
    'rmse',
    'density_kl_0_01', 'density_kl_0_1', 'density_kl_1_0',
    'mrre_zx_10', 'mrre_xz_10', 'mrre_zx_50', 'mrre_xz_50',
}

# ── 1. Load all metrics ───────────────────────────────────────────────────
all_metrics = {}
for dataset in REBUTTAL_DATASETS:
    for model in REBUTTAL_MODELS:
        for dim in REBUTTAL_DIMS:
            per_seed = {}
            for seed in REBUTTAL_SEEDS:
                pattern = f"{REBUTTAL_DIR}/dim{dim}/seed{seed}/{dataset}/{model}/*/metrics.json"
                files = sorted(glob.glob(pattern))
                if not files:
                    continue
                with open(files[-1]) as f:
                    m = json.load(f)
                for metric, val in m.items():
                    if isinstance(val, (int, float)) and '_std' not in metric:
                        per_seed.setdefault(metric, []).append(float(val))
            if per_seed:
                all_metrics[(dataset, model, dim)] = per_seed

# ── 2. Aggregate mean per (dataset, model, dim, metric) ──────────────────
agg = {}
metrics_found = set()
for (dataset, model, dim), metric_dict in all_metrics.items():
    for metric, vals in metric_dict.items():
        vals = [v for v in vals if not np.isnan(v)]
        if vals:
            agg[(dataset, model, dim, metric)] = np.mean(vals)
            metrics_found.add(metric)

# ── 3. Count MMAE wins per metric ────────────────────────────────────────
win_counts     = {m: 0 for m in metrics_found}
total_contests = {m: 0 for m in metrics_found}

for dataset in REBUTTAL_DATASETS:
    for dim in REBUTTAL_DIMS:
        for metric in metrics_found:
            scores = {mod: agg.get((dataset, mod, dim, metric)) for mod in REBUTTAL_MODELS}
            scores = {mod: v for mod, v in scores.items() if v is not None}
            if len(scores) < 2 or 'mmae' not in scores:
                continue
            total_contests[metric] += 1
            winner = min(scores, key=scores.__getitem__) if metric in LOWER_IS_BETTER                      else max(scores, key=scores.__getitem__)
            if winner == 'mmae':
                win_counts[metric] += 1

# ── 4. Win-rate table (ranked) ───────────────────────────────────────────
win_df = pd.DataFrame([
    {'metric': m,
     'mmae_wins': win_counts[m],
     'total': total_contests[m],
     'win_rate_%': round(100 * win_counts[m] / total_contests[m], 1) if total_contests[m] else 0,
     'direction': 'lower=better' if m in LOWER_IS_BETTER else 'higher=better'}
    for m in metrics_found if total_contests[m] > 0
]).sort_values('win_rate_%', ascending=False).reset_index(drop=True)

print(f"MMAE win rates (out of {len(REBUTTAL_DATASETS) * len(REBUTTAL_DIMS)} dataset×dim combos each)
")
display(win_df)

# ── 5. Full mean-only table for a chosen metric ──────────────────────────
# Set to any metric name from win_df above
DISPLAY_METRIC = 'distance_correlation'

rows = []
for dataset in REBUTTAL_DATASETS:
    row = {'dataset': dataset}
    for dim in REBUTTAL_DIMS:
        for model in REBUTTAL_MODELS:
            vals = [v for v in all_metrics.get((dataset, model, dim), {}).get(DISPLAY_METRIC, [])
                    if not np.isnan(v)]
            row[f'{model}_{dim}d'] = f'{np.mean(vals):.3f}' if vals else 'N/A'
    rows.append(row)

tbl = pd.DataFrame(rows).set_index('dataset')
cols = [f'{m}_{d}d' for d in REBUTTAL_DIMS for m in REBUTTAL_MODELS]
tbl = tbl[[c for c in cols if c in tbl.columns]]

direction = 'lower=better' if DISPLAY_METRIC in LOWER_IS_BETTER else 'higher=better'
print(f"
--- {DISPLAY_METRIC} ({direction}) ---")
display(tbl)

# ── 6. LaTeX ─────────────────────────────────────────────────────────────
latex = []
latex.append(r'egin{tabular}{l' + 'c' * len(tbl.columns) + r'}')
latex.append(r'	oprule')
latex.append(' & '.join(['Dataset'] + list(tbl.columns)) + r' \')
latex.append(r'\midrule')
for idx, r in tbl.iterrows():
    latex.append(' & '.join([idx] + list(r.values)) + r' \')
latex.append(r'ottomrule')
latex.append(r'\end{tabular}')
print('
--- LaTeX ---')
print('
'.join(latex))

# Save
save_dir = f'{DRIVE_RESULTS}/rebuttal_w0'
os.makedirs(save_dir, exist_ok=True)
tbl.to_csv(f'{save_dir}/{DISPLAY_METRIC}_table.csv')
with open(f'{save_dir}/{DISPLAY_METRIC}_table.tex', 'w') as f:
    f.write('
'.join(latex))
print(f'
Saved to {save_dir}/')
